# 🦙 Agentic RAG with LlamaIndex

LlamaIndex makes it incredibly easy to turn query engines into tools for an agent. Here we use the `ReActAgent`.

In [ ]:
import os
import nest_asyncio

nest_asyncio.apply()
os.environ['GEMINI_API_KEY'] = os.environ.get('GEMINI_API_KEY', 'YOUR_API_KEY')

In [ ]:
from llama_index.core import Settings, SimpleDirectoryReader, VectorStoreIndex
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.llm = GoogleGenAI(model='gemini-2.5-flash')
Settings.embed_model = HuggingFaceEmbedding(model_name='BAAI/bge-small-en-v1.5')

In [ ]:
print('Loading Data...')
documents = SimpleDirectoryReader('../data').load_data()
print('Building Index...')
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine(similarity_top_k=3)

In [ ]:
from llama_index.core.tools import QueryEngineTool, ToolMetadata

vector_tool = QueryEngineTool(
    query_engine=query_engine,
    metadata=ToolMetadata(
        name='apple_env_report',
        description='Useful for answering questions about Apple\'s environmental goals and progress.'
    )
)

In [ ]:
from llama_index.core.agent import ReActAgent

agent = ReActAgent.from_tools([vector_tool], llm=Settings.llm, verbose=True)

response = agent.chat('What are Apple\'s goals for 2030 and how are they achieving them?')
print(response)